# Multimodal Question Answering (VQA) - Colab Training (H100 AUTOPILOT, resume-safe)

Fine-tunes the **ViLT VQA core** (`dandelin/vilt-b32-finetuned-vqa`) on VQAv2 with HF Trainer
(soft-target BCE, selected on VQA accuracy). The question-type router, the calibrated-abstention
gate and the answer constraints are algorithmic - only the VQA model is trained.

**How to use:** set the controls in cell 0, then **Runtime -> Run all**. Resume-safe (re-run cell 9).
Auto-adapts H100/A100/L4/T4. ViLT fine-tunes even on a free T4.

> Low-confidence answers are returned as 'unsure' and flagged for human review.


In [ ]:
#@title 0) Controls - set these, then `Runtime -> Run all`  { display-mode: "form" }
GIT_REPO_URL = "https://github.com/<your-username>/mmqa"  #@param {type:"string"}
GIT_BRANCH   = "main"  #@param {type:"string"}
USE_DRIVE    = True     #@param {type:"boolean"}
DRIVE_SUBDIR = "mmqa"  #@param {type:"string"}

VQA_BASE     = "dandelin/vilt-b32-finetuned-vqa"  #@param ["dandelin/vilt-b32-finetuned-vqa", "dandelin/vilt-b32-mlm"]
VQA_DATASET  = "HuggingFaceM4/VQAv2"  #@param {type:"string"}
EVAL_DATASET = "lmms-lab/VQAv2"  #@param {type:"string"}
MAX_TRAIN_SAMPLES = 40000  #@param {type:"integer"}
EPOCHS       = 4     #@param {type:"integer"}
RUN_AUTOPILOT = True  #@param {type:"boolean"}
HF_TOKEN     = ""      #@param {type:"string"}
print('Controls set. VQA =', VQA_BASE, '| train =', VQA_DATASET)


In [ ]:
#@title 1) Check the GPU
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'No GPU - Runtime>Change runtime type>GPU')


In [ ]:
#@title 2) Mount Drive + artifact paths & HF caches  (BEFORE importing torch)
import os
ART = '/content/artifacts'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        ART = f'/content/drive/MyDrive/{DRIVE_SUBDIR}/artifacts'
    except Exception as e:
        print('Drive mount skipped:', e)
os.makedirs(ART, exist_ok=True)
os.environ['MMQA_ARTIFACTS_DIR'] = ART
os.environ['HF_HOME'] = f'{ART}/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN; os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
print('Artifacts ->', ART)


In [ ]:
#@title 3) Get the project source (git clone, or copy from Drive)
import os
os.chdir('/content')
if os.path.isdir('/content/mmqa'):
    os.chdir('/content/mmqa'); os.system('git pull')
elif GIT_REPO_URL and '<your-username>' not in GIT_REPO_URL:
    os.system(f'git clone -b {GIT_BRANCH} {GIT_REPO_URL} /content/mmqa'); os.chdir('/content/mmqa')
else:
    drive_src = f'/content/drive/MyDrive/{DRIVE_SUBDIR}/mmqa'
    if os.path.isdir(drive_src):
        os.system(f'cp -r {drive_src} /content/mmqa'); os.chdir('/content/mmqa')
    else:
        raise SystemExit('Set GIT_REPO_URL to your repo, or upload the project to Drive at ' + drive_src)
print('cwd =', os.getcwd()); print(sorted(os.listdir('.'))[:20])


In [ ]:
#@title 4) Install dependencies (Colab-safe: NEVER reinstall torch)
!pip -q install -r requirements_colab.txt
!pip -q install -e . --no-deps
print('deps installed')


In [ ]:
#@title 5) Verify environment + performance knobs (TF32)
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print('GPU:', torch.cuda.get_device_name(0))
import mmqa, transformers, datasets
print('mmqa', mmqa.__version__, '| transformers', transformers.__version__)


In [ ]:
#@title 6) Auto GPU profile (ViLT VQA batch + precision)
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
n = name.upper()
if 'H100' in n:     BATCH = 64
elif 'A100' in n:   BATCH = 32
elif 'L4' in n:     BATCH = 16
elif 'T4' in n:     BATCH = 12
else:               BATCH = 4
BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
FP16 = (not BF16) and torch.cuda.is_available()
TF32 = ('H100' in n or 'A100' in n or 'L4' in n)
print(f'GPU={name} -> batch={BATCH} precision={"bf16" if BF16 else ("fp16" if FP16 else "fp32")}')


In [ ]:
#@title 7) Write the Colab training config  (configs/train_colab.yaml)
import yaml, os
cfg = {
  'project_title': 'Multimodal Question Answering (VQA) System', 'author': 'Le Dinh Minh Quan', 'student_id': '23127460',
  'data': {'vqa_dataset': VQA_DATASET, 'trust_remote_code': True, 'vqa_eval_dataset': EVAL_DATASET,
           'vqa_eval_split': 'validation', 'use_hf': True, 'max_train_samples': int(MAX_TRAIN_SAMPLES),
           'max_eval_samples': 4000, 'synth_eval_scenes': 120, 'image_size': 384, 'seed': 42},
  'model': {'base_model': VQA_BASE, 'model_type': 'vilt', 'image_size': 384, 'max_question_length': 40,
            'num_answers': 3129, 'top_k': 5, 'num_train_epochs': int(EPOCHS), 'learning_rate': 5.0e-5,
            'per_device_train_batch_size': int(BATCH), 'warmup_ratio': 0.1, 'early_stopping_patience': 3,
            'bf16': bool(BF16), 'fp16': bool(FP16), 'tf32': bool(TF32), 'eval_steps': 500, 'save_steps': 500},
  'agent': {'confidence_min': 0.20, 'margin_min': 0.03, 'entropy_max': 2.5,
            'abstain_enabled': True, 'type_consistency_enabled': True},
}
os.makedirs('configs', exist_ok=True)
yaml.safe_dump(cfg, open('configs/train_colab.yaml','w'), sort_keys=False)
print(open('configs/train_colab.yaml').read())


In [ ]:
#@title 8) Render synthetic scenes + sanity-check the datasets (streaming probe)
!PYTHONPATH=src python -m mmqa.cli --config configs/train_colab.yaml gen-synthetic
!PYTHONPATH=src python -m mmqa.cli --config configs/train_colab.yaml data


## ONE BUTTON - autopilot (resume-safe)
Persists the prior baseline, fine-tunes the ViLT VQA core, evaluates VQA accuracy + per-type +
the agent's abstention/coverage, runs analysis, and writes **report.pdf + slides.pptx + grading +
bundle**. Re-run to resume from the last checkpoint.


In [ ]:
#@title 9) ONE BUTTON autopilot  (re-run to resume)
import os
if RUN_AUTOPILOT:
    os.system('PYTHONPATH=src python -m mmqa.cli --config configs/train_colab.yaml autopilot '
              f'--limit {int(MAX_TRAIN_SAMPLES)}')
else:
    print('RUN_AUTOPILOT is off - use the individual steps below.')


## Individual steps (optional) - idempotent + resume-safe


In [ ]:
#@title 10a) Fine-tune the ViLT VQA core (resumes from the last checkpoint)
!PYTHONPATH=src python -m mmqa.cli --config configs/train_colab.yaml train-vqa --limit $MAX_TRAIN_SAMPLES --base-model "$VQA_BASE"


In [ ]:
#@title 10b) Baseline + evaluate (VQA accuracy + per-type + agent coverage) + tune
!PYTHONPATH=src python -m mmqa.cli --config configs/train_colab.yaml train-baseline
!PYTHONPATH=src python -m mmqa.cli --config configs/train_colab.yaml evaluate
!PYTHONPATH=src python -m mmqa.cli --config configs/train_colab.yaml tune


In [ ]:
#@title 11) Diagnostics: eval headline + model metadata
import json, glob, os
rd = os.path.join(os.environ['MMQA_ARTIFACTS_DIR'], 'runs', 'eval.json')
if os.path.exists(rd):
    print(json.dumps(json.load(open(rd)).get('headline', {}), indent=2))
for m in glob.glob(os.path.join(os.environ['MMQA_ARTIFACTS_DIR'], 'models', 'vqa', '*', 'model_meta.json')):
    print(m); print(json.dumps(json.load(open(m)), indent=2)[:500])


## Test the trained model


In [ ]:
#@title 12) Ask a question about the sample image with the trained model
from PIL import Image
import matplotlib.pyplot as plt
!PYTHONPATH=src python -m mmqa.cli --config configs/train_colab.yaml ask --image sample_data/sample_scene.png --question "how many shapes are there?"
plt.imshow(Image.open('sample_data/sample_scene.png')); plt.axis('off'); plt.title('sample image'); plt.show()


In [ ]:
#@title 13) Locate deliverables (report.pdf + slides.pptx + bundle)
import glob, os
base = os.environ['MMQA_ARTIFACTS_DIR']
for pat in ['submission/*/report.pdf', 'submission/*/slides.pptx', 'submission/*/submission_bundle.zip']:
    for f in glob.glob(os.path.join(base, pat)):
        print(round(os.path.getsize(f)/1024, 1), 'KB', f)


In [ ]:
#@title 14) (Optional) Serve the API + Gradio demo
# !PYTHONPATH=src python -m mmqa.cli --config configs/infer.yaml serve --ui --port 7860
print('Uncomment to serve. On Colab add a tunnel (e.g. cloudflared) to expose :7860.')


## Final checklist
- [ ] GPU profile picked a sensible batch/precision
- [ ] `train-vqa` wrote `models/vqa/<version>/`; `train-baseline` wrote `prior_baseline.json`
- [ ] `evaluate` shows the fine-tuned ViLT beating the blind question-only prior on **VQA accuracy**
- [ ] per-answer-type accuracy (yes/no, number, other) looks reasonable + abstention rate is sane
- [ ] `ask` returns a sensible answer with a confidence; low-confidence -> 'unsure'
- [ ] `report.pdf` + `slides.pptx` + `submission_bundle.zip` exist under `artifacts/submission/`
- [ ] Remember: VQAv2 train mirror needs trust_remote_code; LLaVA/Qwen2.5-VL-3B are non-commercial (flagged)
